# 01 · AKUDENTAL Panoramic WSP Benchmark (Validation Only)

Self-contained notebook using the official public AKUDENTAL folder (`akudental_instances.json` + `images/`, stratified **200 train / 67 val / 66 sealed test**).

- **Cell 2 (`akudental_cls` · Stage 1):** Frozen WSP (`K = 0`, 6 locked replies) on the 67 validation films.
- **Cell 3 (`akudental_cls` · Stages 2 & 3):** Random-Init Adam vs. WSP Warm-Start Adam across `K = 2, 4, 8, 16, 32, 64, 200` (5 seeds), saving CSVs and plotting the 2-panel learning curve.
- **Cell 4 (`akudental_seg` · Stage 1):** Frozen `pano-wsp82-v1` segmentation audit (6 locked replies).
- **Cell 5 (`akudental_seg` · Stages 2 & 3):** Converged 82-param comparison (mini-batch + cosine LR + alive ReLU init) across `K = 1..200` (full-train ceiling).
- **Cell 6 (`akudental_seg` · Visual Overlays):** Side-by-side deterministic X-ray comparison figures across all 6 frozen replies and across all 4 arms.

**Governance:** Held-out test split (`66` films) stays sealed (`split='val'`). Clear outputs before committing.

In [ ]:
from pathlib import Path
import sys

# Auto-detect Colab vs. local Windows environment
if Path("/content").is_dir():
    from google.colab import drive
    drive.mount("/content/drive", readonly=True)
    DATA_DIR = Path("/content/drive/MyDrive/WSP_Dental_Data/AKUDENTAL")
    candidates = [Path("/content/dental-wsp"), Path("/content/drive/MyDrive/dental-wsp")]
    REPO_DIR = next((p for p in candidates if (p / "benchmarks").is_dir()), candidates[1])
    OUT_ROOT = Path("/content/runs")
else:
    REPO_DIR = Path(r"C:\Users\Lenovo\Desktop\dental-wsp")
    DATA_DIR = Path(r"C:\Users\Lenovo\Desktop\WSP_Dental_Data\AKUDENTAL")
    OUT_ROOT = REPO_DIR / "runs"

assert (REPO_DIR / "benchmarks").is_dir(), (
    f"dental-wsp code folder not found at {REPO_DIR}. "
    "Upload C:/Users/Lenovo/Desktop/dental-wsp to My Drive/dental-wsp (or clone to /content/dental-wsp)."
)
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

import torch
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

from benchmarks.akudental.adapter import verify_akudental_source
from wsp_core.guards import exposure_guard
exposure_guard(DATA_DIR)
print(f"Repo:   {REPO_DIR}")
print(f"Data:   {DATA_DIR}")
print(f"Device: {DEVICE}")
print(verify_akudental_source(DATA_DIR))

In [ ]:
# Stage 1 · akudental_cls Frozen WSP (K = 0) on 67 validation films
import pandas as pd
from benchmarks.akudental.adapter import load_akudental_cls_public
from wsp_core.evaluation.classification import evaluate_classifier
from wsp_core.schemas import strict_json_loads, verify_response_lock

train_cls = load_akudental_cls_public(DATA_DIR, split="train")
val_cls = load_akudental_cls_public(DATA_DIR, split="val")
print(f"akudental_cls cohorts: train={len(train_cls)} (pos={sum(s.label for s in train_cls)}), val={len(val_cls)} (pos={sum(s.label for s in val_cls)})")

design_path = REPO_DIR / "benchmarks/akudental/locks/akudental_cls.design.lock.json"
resp_path = REPO_DIR / "benchmarks/akudental/locks/akudental_cls.responses.lock.json"
design_bytes = design_path.read_bytes()
frozen_cls = verify_response_lock(strict_json_loads(design_bytes), design_bytes, strict_json_loads(resp_path.read_bytes()), "cls-lenet2d-v1")

stage1_rows = [{"Reply": slot, **evaluate_classifier(p, val_cls)} for slot, p in frozen_cls.items()]
display(pd.DataFrame(stage1_rows).set_index("Reply")[["roc_auc", "pr_auc", "accuracy", "sensitivity", "specificity", "margin_ablated_roc_auc", "sign_flip_flagged"]].round(4))

In [ ]:
# Stages 2 & 3 · akudental_cls Random-Init Adam vs. WSP Warm-Start Adam (K = 2..200, 5 seeds)
import matplotlib.pyplot as plt
from wsp_core.training.runner import plot_classification_learning_curve, run_classification_three_arm_validation

OUT_DIR = OUT_ROOT / "akudental_cls"
BUDGETS = [2, 4, 8, 16, 32, 64, len(train_cls)]
SEEDS = [0, 1, 2, 3, 4]

runs_df, summary_df = run_classification_three_arm_validation(
    task="akudental_cls",
    train_samples=train_cls,
    val_samples=val_cls,
    design_lock_path=design_path,
    responses_lock_path=resp_path,
    budgets=BUDGETS,
    seeds=SEEDS,
    epochs=200,
    lr=5e-3,
    device=DEVICE,
    out_dir=OUT_DIR,
)
display(summary_df.round(4))
fig = plot_classification_learning_curve(summary_df, task="akudental_cls")
plt.show()
plt.close(fig)
print(f"Saved runs, summary CSV, and PNG to {OUT_DIR}")

In [ ]:
# Stage 1 · akudental_seg Frozen WSP (self-contained after Cell 1)
import pandas as pd
from benchmarks.akudental.adapter import load_akudental_seg_public
from wsp_core.evaluation.segmentation import evaluate_segmentation_paired_audit, evaluate_segmentation_swaps
from wsp_core.schemas import strict_json_loads, verify_response_lock

seg_design = REPO_DIR / "benchmarks/akudental/locks/akudental_seg.design.lock.json"
seg_resp = REPO_DIR / "benchmarks/akudental/locks/akudental_seg.responses.lock.json"

if seg_resp.is_file():
    val_seg = load_akudental_seg_public(DATA_DIR, split="val")
    d_bytes = seg_design.read_bytes()
    frozen_seg = verify_response_lock(strict_json_loads(d_bytes), d_bytes, strict_json_loads(seg_resp.read_bytes()), "pano-wsp82-v1")
    rows = []
    for slot, payload in frozen_seg.items():
        sw = evaluate_segmentation_swaps(payload, val_seg)
        au = evaluate_segmentation_paired_audit(payload, val_seg)
        rows.append({
            "Reply": slot,
            "Intact Dice (%)": sw["full_cohort_dice_a_pct"],
            "Image-only Dice (%)": sw["full_cohort_dice_b_pct"],
            "Position-only Dice (%)": sw["full_cohort_dice_c_pct"],
            "Swap loss (pp)": sw["intact"]["swap_loss_pp"],
            "Swap groups": f"{sw['groups']}/{sw['total_groups']}",
            "Outside FP share (%)": au["outside_fp_reduction_share_pct"],
            "Delta edge recall (pp)": au["edge_recall_gain_pp"],
            "Box IoU A (%)": au["box_iou_a_pct"],
            "Excluded tooth pixels A (%)": au["excluded_tooth_a_pct"],
        })
    display(pd.DataFrame(rows).set_index("Reply").round(4))
else:
    print("Waiting for akudental_seg.responses.lock.json (run dental-wsp lock-reply after collecting prompts).")

In [ ]:
# Stages 2 & 3 · akudental_seg 82-Param Frozen WSP vs. Position-Only vs. Random-Init vs. WSP Warm-Start
import matplotlib.pyplot as plt
import pandas as pd
from wsp_core.training.runner import plot_segmentation_learning_curve, run_segmentation_three_arm_validation

if seg_resp.is_file():
    train_seg = load_akudental_seg_public(DATA_DIR, split="train")
    SEG_OUT = OUT_ROOT / "akudental_seg_converged"
    seg_runs, seg_summary = run_segmentation_three_arm_validation(
        task="akudental_seg",
        architecture="pano-wsp82-v1",
        train_samples=train_seg,
        val_samples=val_seg,
        design_lock_path=seg_design,
        responses_lock_path=seg_resp,
        budgets=[1, 2, 4, 8, 16, 32, 64, len(train_seg)],
        seeds=[0, 1, 2],
        epochs=300,
        lr=3e-2,
        batch_size=16,
        cosine_lr=True,
        device=DEVICE,
        out_dir=SEG_OUT,
        resume=True,
    )
    display(seg_summary.round(4))
    fig = plot_segmentation_learning_curve(seg_summary, task="akudental_seg")
    plt.show()
    plt.close(fig)
    print(f"Saved segmentation runs, summary CSV, representative payloads, and PNG to {SEG_OUT}")

In [ ]:
# Cell 6 · Deterministic X-Ray Overlay Comparisons (All 6 Frozen Replies + 4-Arm Comparison)
import json
import matplotlib.pyplot as plt
from wsp_core.visualization import (
    plot_four_arm_comparison,
    plot_frozen_replies_grid,
    select_deterministic_examples,
)

ATTR = "Dataset: AKUDENTAL (Oz et al., BMC Oral Health 2025, CC BY-NC-SA 4.0) · Deterministic validation sample"
examples = select_deterministic_examples(val_seg, n=2)
VIZ_DIR = OUT_ROOT / "akudental_seg_visuals"

for idx, ex in enumerate(examples, start=1):
    fig_a = plot_frozen_replies_grid(
        ex,
        frozen_seg,
        title=f"AKUDENTAL Validation Example {idx} · All 6 Frozen pano-wsp82-v1 Replies (A Intact vs. B Image-Only)",
        attribution=ATTR,
        out_path=VIZ_DIR / f"example_{idx}_frozen_6replies.png",
    )
    img_a = plt.imread(VIZ_DIR / f"example_{idx}_frozen_6replies.png")
    plt.figure(figsize=(11, 12)); plt.imshow(img_a); plt.axis("off"); plt.show()

rep_path = OUT_ROOT / "akudental_seg_converged/akudental_seg_representative_payloads.json"
if rep_path.is_file():
    rep_payloads = json.loads(rep_path.read_text(encoding="utf-8"))
    for idx, ex in enumerate(examples, start=1):
        plot_four_arm_comparison(
            ex,
            rep_payloads,
            title=f"AKUDENTAL Validation Example {idx} · 4-Arm Comparison (82-Parameter pano-wsp82-v1)",
            attribution=ATTR,
            out_path=VIZ_DIR / f"example_{idx}_four_arms.png",
        )
        img_b = plt.imread(VIZ_DIR / f"example_{idx}_four_arms.png")
        plt.figure(figsize=(12, 7)); plt.imshow(img_b); plt.axis("off"); plt.show()
    print(f"Saved X-ray comparison overlays to {VIZ_DIR}")